In [1]:
# 1. Importar todas las librerías necesarias
import pandas as pd
import re
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.parse import unquote, quote_plus

# 2. Cargar el dataset
# Nota: Si tu notebook está dentro de una subcarpeta (como 'notebooks/'), deja '../' para subir a la raíz.
# Si tu notebook está en la misma carpeta principal del proyecto, cambia a: "data/procesado/amazon_electronics_data[(cleaned)].csv"
ruta_csv = Path("../data/procesado/amazon_electronics_data[(cleaned)].csv")
df = pd.read_csv(ruta_csv)

print(f"Dataset cargado con éxito: {df.shape[0]:,} filas | {df.shape[1]} columnas")

# 3. Función para extraer el ASIN de la URL de Amazon
def extraer_asin(url):
    if pd.isna(url):
        return pd.NA

    url = unquote(str(url))
    resultado = re.search(
        r"/(?:dp|gp/product)/([A-Z0-9]{10})(?=[/?#]|$)",
        url,
        flags=re.I
    )

    return resultado.group(1).upper() if resultado else pd.NA

# Aplicar la extracción de ASIN
df["asin"] = df["product_page_url"].apply(extraer_asin)

# 4. Agrupar productos sin ASIN (pendientes)
pendientes = (
    df[df["asin"].isna()]
    .groupby(["product_title", "product_image_url"], dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

# 5. Crear enlace de búsqueda en Amazon de forma segura (evitando errores con nulos)
pendientes["buscar_amazon"] = pendientes["product_title"].apply(
    lambda x: "https://www.amazon.com/s?k=" + quote_plus(str(x)) if pd.notna(x) else ""
)

# 6. Mostrar estadísticas y resultados
print("ASIN identificados:", df["asin"].notna().sum())
print("Productos pendientes:", len(pendientes))

display(pendientes.head(10))

Dataset cargado con éxito: 42,675 filas | 17 columnas
ASIN identificados: 40606
Productos pendientes: 189


,product_title,product_image_url,registros,buscar_amazon
62,"Duracell Coppertop 9V Battery, 6 Count (Pack o...",https://m.media-amazon.com/images/I/61ZSfHGJ2b...,744,https://www.amazon.com/s?k=Duracell+Coppertop+...
129,"Energizer MAX AA Batteries (16 Pack), Double A...",https://m.media-amazon.com/images/I/91wM+1e9MS...,566,https://www.amazon.com/s?k=Energizer+MAX+AA+Ba...
32,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V ...,https://m.media-amazon.com/images/I/71O2yL5jhk...,204,https://www.amazon.com/s?k=CR+2016+MAXELL+LITH...
111,"Energizer Alkaline Power AAA Batteries, 32 Cou...",https://m.media-amazon.com/images/I/91ass+ZYjc...,93,https://www.amazon.com/s?k=Energizer+Alkaline+...
76,Duracell Precharged Recharg. Battery AA (Pack ...,https://m.media-amazon.com/images/I/71Mb2JgtSr...,79,https://www.amazon.com/s?k=Duracell+Precharged...
186,Renata Watch Battery 321 0%Hg Mercury Free X 3,https://m.media-amazon.com/images/I/61uoafSNsv...,79,https://www.amazon.com/s?k=Renata+Watch+Batter...
188,"Windex Electronic Cleaning Wipes, Pre-Moistene...",https://m.media-amazon.com/images/I/81L6As19KP...,65,https://www.amazon.com/s?k=Windex+Electronic+C...
120,Energizer CR2032 Lithium 3V Coin Batteries for...,https://m.media-amazon.com/images/I/81Fk-a9gVK...,19,https://www.amazon.com/s?k=Energizer+CR2032+Li...
133,"Energizer MAX AAA Batteries (16 Pack), Triple ...",https://m.media-amazon.com/images/I/81lUVq-gPX...,14,https://www.amazon.com/s?k=Energizer+MAX+AAA+B...
104,"Energizer AA and AAA Batteries, 48 Count, Comb...",https://m.media-amazon.com/images/I/91+6hAElRP...,6,https://www.amazon.com/s?k=Energizer+AA+and+AA...


In [2]:
# Buscar coincidencias exactas de título e imagen
# entre productos pendientes e identificados

claves = ["product_title", "product_image_url"]

catalogo = (
    df[df["asin"].notna()]
    .groupby(claves, dropna=False)["asin"]
    .agg(
        asin_distintos="nunique",
        asin_posible="first"
    )
    .reset_index()
)

coincidencias = pendientes.merge(
    catalogo,
    on=claves,
    how="left"
)

# Coincidencias con un único ASIN asociado
recuperables = coincidencias[
    coincidencias["asin_distintos"] == 1
]

print("Grupos con coincidencia exacta:", len(recuperables))

print(
    "Registros potencialmente recuperables:",
    recuperables["registros"].sum()
)

display(
    recuperables[
        ["product_title", "registros", "asin_posible"]
    ].head(20)
)

Grupos con coincidencia exacta: 1
Registros potencialmente recuperables: 1


,product_title,registros,asin_posible
40,Renata 371 SR920SW Batteries - 1.55V Silver Ox...,1,B00MU3FWUS


In [3]:
# Buscar coincidencias por título, sin exigir la misma imagen

catalogo_titulos = (
    df[df["asin"].notna()]
    .groupby("product_title")["asin"]
    .agg(
        asin_distintos="nunique",
        asin_posible="first"
    )
    .reset_index()
)

# Excluir el grupo que ya identificamos
restantes = coincidencias.loc[
    coincidencias["asin_distintos"] != 1,
    ["product_title", "product_image_url", "registros"]
]

revision_titulos = restantes.merge(
    catalogo_titulos,
    on="product_title",
    how="left"
)

candidatos = revision_titulos[
    revision_titulos["asin_distintos"] == 1
]

print("Grupos con título coincidente:", len(candidatos))

print(
    "Registros potencialmente recuperables:",
    candidatos["registros"].sum()
)

display(
    candidatos[
        ["product_title", "registros", "asin_posible"]
    ].head(20)
)

Grupos con título coincidente: 1
Registros potencialmente recuperables: 1


,product_title,registros,asin_posible
134,"ACDelco 48-Count Triple AAA Batteries, Maximum...",1,B00FF8KQJC


In [4]:
pd.set_option("display.max_colwidth", None)

display(
    pendientes[
        ["product_title", "registros", "buscar_amazon"]
    ].head(10)
)

,product_title,registros,buscar_amazon
62,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,https://www.amazon.com/s?k=Duracell+Coppertop+9V+Battery%2C+6+Count+%28Pack+of+1%29%2C+9+Volt+Battery+with+Long-lasting+Power%2C+All-Purpose+Alkaline+9V+Battery+for+Household+and+Office+Devices
129,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,https://www.amazon.com/s?k=Energizer+MAX+AA+Batteries+%2816+Pack%29%2C+Double+A+Alkaline+Batteries
32,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,https://www.amazon.com/s?k=CR+2016+MAXELL+LITHIUM+BATTERIES+%282+piece%29+3V+Watch+2016+New
111,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,https://www.amazon.com/s?k=Energizer+Alkaline+Power+AAA+Batteries%2C+32+Count%2C+Long-Lasting+Triple+A+Batteries%2C+Suitable+for+Everyday+Electronics+and+Emergency+Gear
76,Duracell Precharged Recharg. Battery AA (Pack of 2),79,https://www.amazon.com/s?k=Duracell+Precharged+Recharg.+Battery+AA+%28Pack+of+2%29
186,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,https://www.amazon.com/s?k=Renata+Watch+Battery+321+0%25Hg+Mercury+Free+X+3
188,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65,https://www.amazon.com/s?k=Windex+Electronic+Cleaning+Wipes%2C+Pre-Moistened+Screen+Wipes+for+TVs%2C+Phones%2C+and+Other+Electronics%2C+Resealable+for+a+Freshness+Lock%2C+25+Count+%28Pack+of+3%29
120,"Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",19,https://www.amazon.com/s?k=Energizer+CR2032+Lithium+3V+Coin+Batteries+for+Watches%2C+AirTags%2C+Key+Fobs%2C+Small+Electronics+%2810+Pack%29
133,"Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries",14,https://www.amazon.com/s?k=Energizer+MAX+AAA+Batteries+%2816+Pack%29%2C+Triple+A+Alkaline+Batteries
104,"Energizer AA and AAA Batteries, 48 Count, Combo Pack Contains 24 Max Double A and 24 Max Triple A Batteries",6,https://www.amazon.com/s?k=Energizer+AA+and+AAA+Batteries%2C+48+Count%2C+Combo+Pack+Contains+24+Max+Double+A+and+24+Max+Triple+A+Batteries


In [5]:
# Tabla auxiliar para validar ASIN recuperados

asin_pendientes_validacion = pd.DataFrame([
    {
        "product_title": "CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New",
        "registros": 204,
        "asin_candidato": "B00IX9OUTM",
        "fuente": "https://www.amazon.com/dp/B00IX9OUTM",
        "metodo": "Búsqueda externa",
        "estado": "Pendiente de validación"
    }
])

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Pendiente de validación


In [6]:
asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B00IX9OUTM",
    "estado"
] = "Validado"

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado


In [7]:
# Recuperar el producto validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["estado"] == "Validado"
].iloc[0]

# Identificar registros pendientes del producto
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

# Verificar cantidad antes de modificar
cantidad = filtro.sum()

print("Registros encontrados:", cantidad)

if cantidad == producto["registros"]:

    # Registrar procedencia del ASIN
    if "asin_origen" not in df.columns:
        df["asin_origen"] = pd.NA
        df.loc[df["asin"].notna(), "asin_origen"] = "URL original"

    # Incorporar identificador validado
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)

else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 204
ASIN incorporado correctamente.
Registros actualizados: 204


In [8]:
# Guardar dataset con ASIN
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print("Archivo guardado:", ruta_salida)
print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())

Archivo guardado: ../data/procesado/amazon_electronics_cleaned_with_asin.csv
Total de registros: 42675
Registros con ASIN: 40810
Registros pendientes: 1865


In [10]:
# Guardar dataset actualizado
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

# Verificar resultados
print("Archivo guardado:", ruta_salida)
print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

Archivo guardado: ../data/procesado/amazon_electronics_cleaned_with_asin.csv
Total de registros: 42675
Registros con ASIN: 40810
Registros pendientes: 1865
ASIN únicos: 8631


In [11]:
nuevo = {
    "product_title": "Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",
    "registros": 744,
    "asin_candidato": "B0B72DBKVF",
    "fuente": "https://www.amazon.com/dp/B0B72DBKVF",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

asin_pendientes_validacion = pd.concat(
    [asin_pendientes_validacion, pd.DataFrame([nuevo])],
    ignore_index=True
)

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)


In [12]:
# Seleccionar el producto Duracell validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B0B72DBKVF"
].iloc[0]

# Identificar registros sin ASIN
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN únicamente si coincide la cantidad esperada
if cantidad == 744:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 744
ASIN incorporado correctamente.
Registros actualizados: 744


In [13]:
# Verificar actualización de los ASIN

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar específicamente Duracell
print(
    "Registros Duracell identificados:",
    (df["asin"] == "B0B72DBKVF").sum()
)

Total de registros: 42675
Registros con ASIN: 41554
Registros pendientes: 1121
ASIN únicos: 8632
Registros Duracell identificados: 744


In [14]:
# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Verificar antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 41554
    and (df["asin"] == "B0B72DBKVF").sum() == 744
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")
else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros con ASIN: 41554
Registros pendientes: 1121
Cobertura: 97.37 %


In [15]:
nuevo = {
    "product_title": "Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",
    "registros": 566,
    "asin_candidato": "B004RFF57C",
    "fuente": "https://www.amazon.com/dp/B004RFF57C",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registrar dos veces el mismo candidato
if not (
    asin_pendientes_validacion["asin_candidato"] == "B004RFF57C"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)


In [16]:
# Seleccionar el producto Energizer validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B004RFF57C"
].iloc[0]

# Identificar registros pendientes del producto
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN solo si coincide la cantidad esperada
if cantidad == 566:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 566
ASIN incorporado correctamente.
Registros actualizados: 566
Total con ASIN: 42120
Registros pendientes: 555


In [17]:
# Verificar actualización del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar específicamente Energizer
print(
    "Registros Energizer identificados:",
    (df["asin"] == "B004RFF57C").sum()
)

# Calcular cobertura
print(
    "Cobertura de ASIN:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42120
Registros pendientes: 555
ASIN únicos: 8633
Registros Energizer identificados: 566
Cobertura de ASIN: 98.7 %


In [18]:
# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42120
    and (df["asin"] == "B004RFF57C").sum() == 566
    and (df["asin"] == "B0B72DBKVF").sum() == 744
    and (df["asin"] == "B00IX9OUTM").sum() == 204
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42120
Registros pendientes: 555
ASIN únicos: 8633
Cobertura: 98.7 %


In [19]:
# Actualizar productos pendientes de identificación

pendientes = (
    df[df["asin"].isna()]
    .groupby("product_title", dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values("registros", ascending=False)
)

print("Registros pendientes:", pendientes["registros"].sum())
print("Títulos pendientes:", len(pendientes))

display(pendientes.head(15))

Registros pendientes: 555
Títulos pendientes: 186


,product_title,registros
109,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93
183,Renata Watch Battery 321 0%Hg Mercury Free X 3,79
74,Duracell Precharged Recharg. Battery AA (Pack of 2),79
185,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65
118,"Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",19
130,"Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries",14
102,"Energizer AA and AAA Batteries, 48 Count, Combo Pack Contains 24 Max Double A and 24 Max Triple A Batteries",6
98,"Energizer AA Batteries Alkaline Power, 32 Count Alkaline Double A Battery Pack",6
107,"Energizer Alkaline Power AA Batteries (20 Pack), Double A Batteries Alkaline, Long-Lasting",4
35,Duracell - 2450 3V Lithium Coin Battery - long lasting battery - 6 Count (Pack of 1),3


In [20]:
nuevo = {
    "product_title": "Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",
    "registros": 93,
    "asin_candidato": "B09RTVN5GN",
    "fuente": "https://www.amazon.com/dp/B09RTVN5GN",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registros duplicados
if not (
    asin_pendientes_validacion["asin_candidato"] == "B09RTVN5GN"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)


In [21]:
# Seleccionar el producto Energizer AAA validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B09RTVN5GN"
].iloc[0]

# Identificar registros pendientes
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN únicamente si coincide la cantidad esperada
if cantidad == 93:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 93
ASIN incorporado correctamente.
Registros actualizados: 93
Total con ASIN: 42213
Registros pendientes: 462


In [22]:
# Verificar integridad del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar Energizer AAA de 32 unidades
print(
    "Registros Energizer AAA identificados:",
    (df["asin"] == "B09RTVN5GN").sum()
)

print(
    "Cobertura:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42213
Registros pendientes: 462
ASIN únicos: 8634
Registros Energizer AAA identificados: 93
Cobertura: 98.92 %


In [23]:
from pathlib import Path

# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42213
    and (df["asin"] == "B09RTVN5GN").sum() == 93
    and (df["asin"] == "B004RFF57C").sum() == 566
    and (df["asin"] == "B0B72DBKVF").sum() == 744
    and (df["asin"] == "B00IX9OUTM").sum() == 204
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42213
Registros pendientes: 462
ASIN únicos: 8634
Cobertura: 98.92 %


In [24]:
nuevo = {
    "product_title": "Renata Watch Battery 321 0%Hg Mercury Free X 3",
    "registros": 79,
    "asin_candidato": "B00NHYB6VI",
    "fuente": "https://www.amazon.com/dp/B00NHYB6VI",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registrar dos veces el mismo candidato
if not (
    asin_pendientes_validacion["asin_candidato"] == "B00NHYB6VI"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)
4,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,B00NHYB6VI,https://www.amazon.com/dp/B00NHYB6VI,Verificación externa Amazon,Validado (ficha actual)


In [25]:
# Seleccionar el producto Renata validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B00NHYB6VI"
].iloc[0]

# Identificar registros pendientes
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN solo si coincide la cantidad esperada
if cantidad == 79:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 79
ASIN incorporado correctamente.
Registros actualizados: 79
Total con ASIN: 42292
Registros pendientes: 383


In [26]:
# Verificar integridad del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar específicamente Renata
print(
    "Registros Renata identificados:",
    (df["asin"] == "B00NHYB6VI").sum()
)

print(
    "Cobertura:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42292
Registros pendientes: 383
ASIN únicos: 8635
Registros Renata identificados: 79
Cobertura: 99.1 %


In [27]:
from pathlib import Path

# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42292
    and (df["asin"] == "B00NHYB6VI").sum() == 79
    and (df["asin"] == "B09RTVN5GN").sum() == 93
    and (df["asin"] == "B004RFF57C").sum() == 566
    and (df["asin"] == "B0B72DBKVF").sum() == 744
    and (df["asin"] == "B00IX9OUTM").sum() == 204
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42292
Registros pendientes: 383
ASIN únicos: 8635
Cobertura: 99.1 %


In [28]:
nuevo = {
    "product_title": "Duracell Precharged Recharg. Battery AA (Pack of 2)",
    "registros": 79,
    "asin_candidato": "B081J2QH83",
    "fuente": "https://www.amazon.com/dp/B081J2QH83",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registros duplicados
if not (
    asin_pendientes_validacion["asin_candidato"] == "B081J2QH83"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)
4,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,B00NHYB6VI,https://www.amazon.com/dp/B00NHYB6VI,Verificación externa Amazon,Validado (ficha actual)
5,Duracell Precharged Recharg. Battery AA (Pack of 2),79,B081J2QH83,https://www.amazon.com/dp/B081J2QH83,Verificación externa Amazon,Validado (ficha actual)


In [29]:
# Seleccionar el producto Duracell validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B081J2QH83"
].iloc[0]

# Identificar registros pendientes
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN solo si coincide la cantidad esperada
if cantidad == 79:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 79
ASIN incorporado correctamente.
Registros actualizados: 79
Total con ASIN: 42371
Registros pendientes: 304


In [31]:
# Verificar integridad del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar Duracell AA recargable
print(
    "Registros Duracell identificados:",
    (df["asin"] == "B081J2QH83").sum()
)

# Calcular cobertura
print(
    "Cobertura:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42371
Registros pendientes: 304
ASIN únicos: 8636
Registros Duracell identificados: 79
Cobertura: 99.29 %


In [32]:
from pathlib import Path

# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42371
    and df["asin"].isna().sum() == 304
    and (df["asin"] == "B081J2QH83").sum() == 79
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42371
Registros pendientes: 304
ASIN únicos: 8636
Cobertura: 99.29 %


In [33]:
nuevo = {
    "product_title": "Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",
    "registros": 65,
    "asin_candidato": "B072B91QTN",
    "fuente": "https://www.amazon.com/dp/B072B91QTN",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registros duplicados
if not (
    asin_pendientes_validacion["asin_candidato"] == "B072B91QTN"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)
4,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,B00NHYB6VI,https://www.amazon.com/dp/B00NHYB6VI,Verificación externa Amazon,Validado (ficha actual)
5,Duracell Precharged Recharg. Battery AA (Pack of 2),79,B081J2QH83,https://www.amazon.com/dp/B081J2QH83,Verificación externa Amazon,Validado (ficha actual)
6,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65,B072B91QTN,https://www.amazon.com/dp/B072B91QTN,Verificación externa Amazon,Validado (ficha actual)


In [34]:
# Seleccionar el producto Windex validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B072B91QTN"
].iloc[0]

# Identificar únicamente los registros pendientes
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar el ASIN si coincide con la cantidad esperada
if cantidad == 65:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 65
ASIN incorporado correctamente.
Registros actualizados: 65
Total con ASIN: 42436
Registros pendientes: 239


In [35]:
# Verificar integridad del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar específicamente Windex
print(
    "Registros Windex identificados:",
    (df["asin"] == "B072B91QTN").sum()
)

# Calcular cobertura
print(
    "Cobertura:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42436
Registros pendientes: 239
ASIN únicos: 8637
Registros Windex identificados: 65
Cobertura: 99.44 %


In [36]:
from pathlib import Path

# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42436
    and df["asin"].isna().sum() == 239
    and (df["asin"] == "B072B91QTN").sum() == 65
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42436
Registros pendientes: 239
ASIN únicos: 8637
Cobertura: 99.44 %


In [37]:
nuevo = {
    "product_title": "Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",
    "registros": 19,
    "asin_candidato": "B0CH1N7X5W",
    "fuente": "https://www.amazon.com/dp/B0CH1N7X5W",
    "metodo": "Verificación externa Amazon",
    "estado": "Validado (ficha actual)"
}

# Evitar registrar dos veces el mismo candidato
if not (
    asin_pendientes_validacion["asin_candidato"] == "B0CH1N7X5W"
).any():
    asin_pendientes_validacion = pd.concat(
        [asin_pendientes_validacion, pd.DataFrame([nuevo])],
        ignore_index=True
    )

display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)
4,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,B00NHYB6VI,https://www.amazon.com/dp/B00NHYB6VI,Verificación externa Amazon,Validado (ficha actual)
5,Duracell Precharged Recharg. Battery AA (Pack of 2),79,B081J2QH83,https://www.amazon.com/dp/B081J2QH83,Verificación externa Amazon,Validado (ficha actual)
6,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65,B072B91QTN,https://www.amazon.com/dp/B072B91QTN,Verificación externa Amazon,Validado (ficha actual)
7,"Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",19,B0CH1N7X5W,https://www.amazon.com/dp/B0CH1N7X5W,Verificación externa Amazon,Validado (ficha actual)


In [38]:
# Seleccionar el producto Energizer CR2032 validado
producto = asin_pendientes_validacion.loc[
    asin_pendientes_validacion["asin_candidato"] == "B0CH1N7X5W"
].iloc[0]

# Identificar registros pendientes
filtro = (
    (df["product_title"] == producto["product_title"]) &
    (df["asin"].isna())
)

cantidad = int(filtro.sum())

print("Registros encontrados:", cantidad)

# Incorporar ASIN únicamente si coincide la cantidad esperada
if cantidad == 19:
    df.loc[filtro, "asin"] = producto["asin_candidato"]
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    print("ASIN incorporado correctamente.")
    print("Registros actualizados:", cantidad)
    print("Total con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
else:
    print("Revisar coincidencias antes de actualizar.")

Registros encontrados: 19
ASIN incorporado correctamente.
Registros actualizados: 19
Total con ASIN: 42455
Registros pendientes: 220


In [39]:
# Verificar integridad del dataset

print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())

# Verificar los registros recuperados de Energizer CR2032
filtro = (
    (df["asin"] == "B0CH1N7X5W") &
    (df["asin_origen"] == "Validación manual Amazon")
)

print("Registros Energizer CR2032 recuperados:", filtro.sum())

print(
    "Cobertura:",
    round(df["asin"].notna().mean() * 100, 2), "%"
)

Total de registros: 42675
Registros con ASIN: 42455
Registros pendientes: 220
ASIN únicos: 8638
Registros Energizer CR2032 recuperados: 19
Cobertura: 99.48 %


In [40]:
from pathlib import Path

# Ruta del archivo de trabajo
ruta_salida = Path(
    "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
)

# Validar integridad antes de guardar
if (
    len(df) == 42675
    and df["asin"].notna().sum() == 42455
    and df["asin"].isna().sum() == 220
    and (
        (df["asin"] == "B0CH1N7X5W") &
        (df["asin_origen"] == "Validación manual Amazon")
    ).sum() == 19
):
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print("Archivo actualizado correctamente.")
    print("Registros totales:", len(df))
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

else:
    print("Verificación no superada. No se guardaron cambios.")

Archivo actualizado correctamente.
Registros totales: 42675
Registros con ASIN: 42455
Registros pendientes: 220
ASIN únicos: 8638
Cobertura: 99.48 %


In [43]:
import pandas as pd
import re
from pathlib import Path

# 1. Producto pendiente
titulo = "Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries"
esperados = 14

# 2. Validación manual
asin_validado = input("Ingresa el ASIN confirmado en Amazon: ").strip().upper()

if not re.fullmatch(r"[A-Z0-9]{10}", asin_validado):
    raise ValueError("El ASIN debe contener exactamente 10 caracteres alfanuméricos.")

confirmacion = input(
    f"¿Confirmas que {asin_validado} corresponde al producto? (SI/NO): "
).strip().upper()

if confirmacion != "SI":
    print("Proceso cancelado. No se modificaron los datos.")

else:
    # 3. Verificar los registros pendientes
    filtro = (df["product_title"] == titulo) & df["asin"].isna()
    cantidad = int(filtro.sum())

    if cantidad != esperados:
        raise ValueError(
            f"Se esperaban {esperados} registros, pero se encontraron {cantidad}. "
            "No se modificaron los datos."
        )

    # 4. Preparar el DataFrame actualizado
    df_nuevo = df.copy()

    if "asin_origen" not in df_nuevo.columns:
        df_nuevo["asin_origen"] = pd.NA

    df_nuevo.loc[filtro, "asin"] = asin_validado
    df_nuevo.loc[filtro, "asin_origen"] = "Validación manual Amazon"

    # 5. Actualizar tabla auxiliar de validaciones
    nuevo = {
        "product_title": titulo,
        "registros": cantidad,
        "asin_candidato": asin_validado,
        "fuente": f"https://www.amazon.com/dp/{asin_validado}",
        "metodo": "Verificación externa Amazon",
        "estado": "Validado (ficha actual)"
    }

    validaciones_nuevas = asin_pendientes_validacion.copy()

    validaciones_nuevas = validaciones_nuevas[
        validaciones_nuevas["product_title"] != titulo
    ]

    validaciones_nuevas = pd.concat(
        [validaciones_nuevas, pd.DataFrame([nuevo])],
        ignore_index=True
    )

    # 6. Guardar archivos
    carpeta = Path("../data/procesado")
    ruta_datos = carpeta / "amazon_electronics_cleaned_with_asin.csv"
    ruta_validaciones = carpeta / "asin_pendientes_validacion.csv"

    if not ruta_datos.is_file():
        raise FileNotFoundError(f"No existe el archivo de trabajo: {ruta_datos}")

    df_nuevo.to_csv(ruta_datos, index=False, encoding="utf-8-sig")
    validaciones_nuevas.to_csv(
        ruta_validaciones, index=False, encoding="utf-8-sig"
    )

    # 7. Actualizar variables en memoria
    df = df_nuevo
    asin_pendientes_validacion = validaciones_nuevas

    # 8. Reporte final
    print("\nASIN incorporado y archivos guardados.")
    print("ASIN validado:", asin_validado)
    print("Registros actualizados:", cantidad)
    print("Registros con ASIN:", df["asin"].notna().sum())
    print("Registros pendientes:", df["asin"].isna().sum())
    print("ASIN únicos:", df["asin"].nunique())
    print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")


ASIN incorporado y archivos guardados.
ASIN validado: B0D51CSMKX
Registros actualizados: 14
Registros con ASIN: 42469
Registros pendientes: 206
ASIN únicos: 8639
Cobertura: 99.52 %


In [45]:

import pandas as pd
from pathlib import Path

# Producto y ASIN candidato
titulo = (
    "Energizer AA and AAA Batteries, 48 Count, "
    "Combo Pack Contains 24 Max Double A and "
    "24 Max Triple A Batteries"
)
asin = "B087GMJXMC"
esperados = 6

# Confirmación manual
confirmacion = input(
    f"¿Confirmas el ASIN {asin} en Amazon? (SI/NO): "
).strip().upper()

if confirmacion != "SI":
    print("Cancelado. No se modificaron los datos.")

else:
    filtro = df["product_title"].eq(titulo) & df["asin"].isna()
    cantidad = int(filtro.sum())

    ruta = Path(
        "../data/procesado/amazon_electronics_cleaned_with_asin.csv"
    )
    ruta_auditoria = Path(
        "../data/procesado/asin_pendientes_validacion.csv"
    )

    if cantidad != esperados:
        print(f"Error: se esperaban {esperados}, encontrados {cantidad}.")

    elif not ruta.is_file():
        print("Error: no existe el archivo de trabajo.")

    else:
        # Incorporación
        df.loc[filtro, "asin"] = asin
        df.loc[filtro, "asin_origen"] = "Validación manual Amazon"

        # Registro de trazabilidad
        nuevo = {
            "product_title": titulo,
            "registros": cantidad,
            "asin_candidato": asin,
            "fuente": f"https://www.amazon.com/dp/{asin}",
            "metodo": "Verificación externa Amazon",
            "estado": "Validado (ficha actual)"
        }

        asin_pendientes_validacion = pd.concat(
            [
                asin_pendientes_validacion[
                    asin_pendientes_validacion["product_title"] != titulo
                ],
                pd.DataFrame([nuevo])
            ],
            ignore_index=True
        )

        # Guardar ambos archivos
        df.to_csv(ruta, index=False, encoding="utf-8-sig")
        asin_pendientes_validacion.to_csv(
            ruta_auditoria, index=False, encoding="utf-8-sig"
        )

        print("Archivos actualizados correctamente.")
        print("Registros incorporados:", cantidad)
        print("Registros con ASIN:", df["asin"].notna().sum())
        print("Pendientes:", df["asin"].isna().sum())
        print("Cobertura:", round(df["asin"].notna().mean()*100, 2), "%")


Archivos actualizados correctamente.
Registros incorporados: 6
Registros con ASIN: 42475
Pendientes: 200
Cobertura: 99.53 %


In [46]:

import pandas as pd
from pathlib import Path

# 1. Datos del producto
titulo = "Energizer AA Batteries Alkaline Power, 32 Count Alkaline Double A Battery Pack"
asin = "B09RTVD1GF"
esperados = 6

# 2. Rutas
carpeta = Path("../data/procesado")
ruta_datos = carpeta / "amazon_electronics_cleaned_with_asin.csv"
ruta_validaciones = carpeta / "asin_pendientes_validacion.csv"

# 3. Confirmación manual
confirmacion = input(
    f"¿Confirmas que el ASIN {asin} corresponde al producto? (SI/NO): "
).strip().upper()

if confirmacion != "SI":
    print("Operación cancelada. No se modificaron datos.")

else:
    # 4. Comprobar registros y archivo
    filtro = df["product_title"].eq(titulo) & df["asin"].isna()
    cantidad = int(filtro.sum())

    if cantidad != esperados:
        print(f"Error: esperados {esperados}, encontrados {cantidad}.")

    elif not ruta_datos.is_file():
        print("Error: no se encontró el CSV de trabajo.")

    else:
        # 5. Preparar incorporación
        df_nuevo = df.copy()
        if "asin_origen" not in df_nuevo.columns:
            df_nuevo["asin_origen"] = pd.NA

        df_nuevo.loc[filtro, "asin"] = asin
        df_nuevo.loc[filtro, "asin_origen"] = "Validación manual Amazon"

        # 6. Registrar trazabilidad
        nuevo = {
            "product_title": titulo,
            "registros": cantidad,
            "asin_candidato": asin,
            "fuente": f"https://www.amazon.com/dp/{asin}",
            "metodo": "Verificación externa Amazon",
            "estado": "Validado (ficha actual)"
        }

        validaciones_nuevas = pd.concat(
            [
                asin_pendientes_validacion.loc[
                    asin_pendientes_validacion["product_title"] != titulo
                ],
                pd.DataFrame([nuevo])
            ],
            ignore_index=True
        )

        # 7. Guardar archivos
        df_nuevo.to_csv(ruta_datos, index=False, encoding="utf-8-sig")
        validaciones_nuevas.to_csv(
            ruta_validaciones, index=False, encoding="utf-8-sig"
        )

        # 8. Actualizar variables en memoria
        df = df_nuevo
        asin_pendientes_validacion = validaciones_nuevas

        print("\nArchivos actualizados correctamente.")
        print("ASIN:", asin)
        print("Registros incorporados:", cantidad)
        print("Registros con ASIN:", df["asin"].notna().sum())
        print("Registros pendientes:", df["asin"].isna().sum())
        print("ASIN únicos:", df["asin"].nunique())
        print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")



Archivos actualizados correctamente.
ASIN: B09RTVD1GF
Registros incorporados: 6
Registros con ASIN: 42481
Registros pendientes: 194
ASIN únicos: 8641
Cobertura: 99.55 %
